Data Loading, Prediction Point Lock

In [3]:
import sys
from pathlib import Path

# Resolve project root containing the 'src' directory
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()),
    Path("..").resolve(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.feature_engineering import InvoiceFeatureEngineer, PipelineConfig

config = PipelineConfig(
    input_path=project_root / "data/processed/cleaned_invoices.parquet",
    processed_dir=project_root / "data/processed",
    reports_dir=project_root / "reports",
    artifacts_dir=project_root / "models",
)

engineer = InvoiceFeatureEngineer(config)
train_df, val_df, test_df = engineer.run()

2026-11-21 13:26:46 | INFO     | Loading cleaned dataset from c:\Users\user\Desktop\InvoiceIQ\data\processed\cleaned_invoices.parquet
2026-11-21 13:27:04 | INFO     | Loaded 2,500,000 rows (2019-01-01 to 2023-12-31)
2026-11-21 13:27:04 | INFO     | Targets defined | paid_late rate: 17.02% | delay_days non-null: 2,074,410
2026-11-21 13:27:05 | INFO     | Task A complete: Engineered time and payment term features.
2026-11-21 13:27:51 | INFO     | Task B complete: Engineered point-in-time customer history.
2026-11-21 13:27:53 | INFO     | Task C complete: Engineered invoice amount and ratio features.
2026-11-21 13:27:59 | INFO     | Split Train | 2,000,456 rows (80.02%) | 2019-01-01 -> 2022-12-31 | Late Rate: 17.02%
2026-11-21 13:27:59 | INFO     | Split Val   |   247,598 rows ( 9.90%) | 2023-01-01 -> 2023-06-30 | Late Rate: 16.96%
2026-11-21 13:27:59 | INFO     | Split Test  |   251,946 rows (10.08%) | 2023-07-01 -> 2023-12-31 | Late Rate: 17.10%
2026-11-21 13:28:17 | INFO     | Task D c

In [4]:
import pandas as pd

summary = pd.DataFrame({
    "Split": ["Train (2019–2022)", "Validation (H1 2023)", "Test (H2 2023)"],
    "Rows": [len(train_df), len(val_df), len(test_df)],
    "Input Features": [len(engineer.feature_cols)] * 3,
    "Late Rate (%)": [
        round(train_df["paid_late"].mean() * 100, 2),
        round(val_df["paid_late"].mean() * 100, 2),
        round(test_df["paid_late"].mean() * 100, 2),
    ],
    "Missing Values": [
        int(train_df[engineer.feature_cols].isna().sum().sum()),
        int(val_df[engineer.feature_cols].isna().sum().sum()),
        int(test_df[engineer.feature_cols].isna().sum().sum()),
    ],
})
display(summary)
display(train_df[engineer.feature_cols[:12]].head(5))

,Split,Rows,Input Features,Late Rate (%),Missing Values
0,Train (2019–2022),2000456,85,17.02,0
1,Validation (H1 2023),247598,85,16.96,0
2,Test (H2 2023),251946,85,17.10,0


,issue_month,issue_quarter,issue_weekday,is_month_end,is_quarter_end,is_first_time_customer,is_net_15,is_net_30,is_net_45,is_net_60,quantity,unit_price
0,1,1,1,0,0,1,0,1,0,0,30,1599.98
1,1,1,1,0,0,1,0,1,0,0,43,2189.05
2,1,1,1,0,0,1,1,0,0,0,19,769.64
3,1,1,1,0,0,1,1,0,0,0,37,32.32
4,1,1,1,0,0,1,1,0,0,0,3,984.17
